# 音声文字起こし・話者分離（汎用版 / Google Colab）

**ファイル名の書き換えは一切不要です。** 音声を選択すると、ファイル名から保存先と成果物名を自動作成します。複数ファイルも順番に処理します。

- 音声認識：**Whisper large-v3（faster-whisper）**／日本語初期設定／単語時刻付き
- 話者分離：**pyannote.audio Community-1**（オン・オフ切替可能）
- 入力：MP3 / M4A / WAV / FLAC / OGG / OPUS / AAC / MP4 / WEBM / MOV など
- 出力：時刻付きTXT・Excel向けCSV・SRT・ASR JSON・話者区間・ZIP
- キャッシュ：音声のSHA-256と設定を照合し、同じ処理を重複実行しない
- データ：音声認識・話者分離はColab上で実行。外部の音声認識APIへのアップロード処理なし（モデルの取得にはHugging Faceへアクセス）

## 使い方
1. **［ランタイム］→［ランタイムのタイプを変更］→［T4 GPU］** を選択する。
2. **セル①**でライブラリをインストールし、**ランタイムを一度再起動**。①のセルを飛ばし、**①-2動作環境チェック**から進む。その後**セル②**の設定を確認。ファイル名の入力は不要。
3. **セル③**を実行して音声を選択する（複数選択可）。
4. 話者分離がオンの場合のみ、Hugging Face Readトークンを求められる（Colabのシークレット `HF_TOKEN` に設定していれば入力不要）。
5. 以降のセルを順に実行すると、最後にZIPを自動ダウンロードする。

**重要**：話者の氏名・役職は自動特定しません。音声認識と話者分離は誤る場合があります。行政内部の録音は所属組織の規程を確認してからColabにアップロードしてください。

## ① インストール（新しいColabランタイムで最初に1回だけ実行）
ライブラリの入れ替えによるNumPyインポートエラーを避けるため、**インストール後に［ランタイム］→［セッションを再起動］を実行**してください。再起動したらこのインストールセルは飛ばし、次の「動作環境チェック」から進みます。


In [ ]:
import sys, os, shutil, subprocess, ctypes, importlib
from pathlib import Path

packages = [
    "faster-whisper==1.2.1",
    "av==18.1.0",
    "pyannote.audio==4.0.7",
    "soundfile>=0.12",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
if shutil.which("ffmpeg") is None:
    subprocess.check_call(["apt-get", "update", "-qq"])
    subprocess.check_call(["apt-get", "install", "-y", "-qq", "ffmpeg"])


_INSTALL_DONE_THIS_SESSION = True
print('インストール完了。ランタイムを再起動し、このセルを飛ばして動作環境チェックから進んでください。')


## ①-2 動作環境チェック（再起動後に実行）
インストール直後にそのまま進むと停止します。Colabのランタイムを再起動し、インストールセルを飛ばしてここから進んでください。


In [ ]:
if globals().get("_INSTALL_DONE_THIS_SESSION", False):
    raise RuntimeError("インストール直後はランタイムを再起動してください。再起動後は①のセルを飛ばしてこのセルから実行します。")

import sys, os, shutil, subprocess, ctypes, importlib
from pathlib import Path
try:
    import av, torch, pyannote.audio, numpy as np
except ImportError as exc:
    raise ImportError("依存ライブラリの読み込みに失敗。インストール後はColabランタイムを再起動してください。") from exc
if av.__version__.split(".")[0] != "18":
    raise RuntimeError("PyAV 18が読み込まれていません。［ランタイム］→［セッションを再起動］してから最初から実行してください。")

if torch.cuda.is_available():
    # ColabのPyTorchがCUDA 13でも、faster-whisper / CTranslate2でCUDA 12ライブラリが必要な場合がある。
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-deps",
        "nvidia-cublas-cu12", "nvidia-cudnn-cu12==9.*",
    ])
    def nvidia_libdir(name):
        module = importlib.import_module(f"nvidia.{name}.lib")
        return Path(next(iter(module.__path__)))
    cublas, cudnn = nvidia_libdir("cublas"), nvidia_libdir("cudnn")
    config_file = Path("/etc/ld.so.conf.d/99-general-whisper-colab.conf")
    config_file.write_text(f"{cublas}\n{cudnn}\n", encoding="utf-8")
    subprocess.run(["ldconfig"], check=True)
    for library in (cublas / "libcublasLt.so.12", cublas / "libcublas.so.12", cudnn / "libcudnn.so.9"):
        ctypes.CDLL(str(library), mode=ctypes.RTLD_GLOBAL)
    print("CUDA 12用cuBLAS/cuDNN: 読み込みOK")
else:
    print("注意：GPUが無効です。［ランタイム］→［ランタイムのタイプを変更］からT4 GPUを選択してください。")

print("PyAV", av.__version__, "/ PyTorch", torch.__version__, "/ CUDA", torch.cuda.is_available(),
      "/ pyannote", getattr(pyannote.audio, "__version__", "4.0.7"))

## ② 任意設定：通常は変更不要
**ファイル名は設定しません。** 話者分離が不要なら `ENABLE_DIARIZATION` をオフにします。会議の実際の話者数が分かる場合、`EXPECTED_SPEAKERS` に人数を入力すると誤分割を減らせる場合があります。

In [ ]:
# @title 文字起こしのオプション（必要な場合のみ変更）
# 0は自動推定/制限なし。GLOSSARYには任意の事業名・地名を読点区切りで入力。
ENABLE_DIARIZATION = True   # @param {type:"boolean"}
EXPECTED_SPEAKERS = 0       # @param {type:"integer"}
MIN_SPEAKERS = 0            # @param {type:"integer"}
MAX_SPEAKERS = 0            # @param {type:"integer"}
LANGUAGE = "ja"             # @param ["ja", "auto", "en"]
WHISPER_MODEL = "large-v3"  # @param ["large-v3", "medium", "small"]
GLOSSARY = ""               # @param {type:"string"}
INPUT_MODE = "upload"       # @param ["upload", "colab_existing"]
SAVE_TO_DRIVE = False       # @param {type:"boolean"}
DRIVE_OUTPUT_FOLDER = "文字起こし結果"  # @param {type:"string"}

if EXPECTED_SPEAKERS < 0 or MIN_SPEAKERS < 0 or MAX_SPEAKERS < 0:
    raise ValueError("話者数の設定には0以上の整数を指定してください。")
if MIN_SPEAKERS and MAX_SPEAKERS and MIN_SPEAKERS > MAX_SPEAKERS:
    raise ValueError("MIN_SPEAKERSはMAX_SPEAKERS以下にしてください。")
if EXPECTED_SPEAKERS and (MIN_SPEAKERS or MAX_SPEAKERS):
    print("注：EXPECTED_SPEAKERSが指定されているためMIN/MAX_SPEAKERSは使いません。")
print("文字起こし:", WHISPER_MODEL, "/ 話者分離:", ENABLE_DIARIZATION,
      "/ 指定話者数:", EXPECTED_SPEAKERS or "自動")

## ③ 音声ファイルの選択（毎回ここで選ぶだけ）
標準では**ファイル選択ダイアログ**が開きます。音声を一つまたは複数選んでください。

Colab左側のファイル欄にドラッグ＆ドロップした音声を使う場合は、②の `INPUT_MODE` を `colab_existing` にしてください。複数ファイルがあれば、その場で番号を選べます。

In [ ]:
from pathlib import Path
from google.colab import files
import hashlib, gc, json, re

AUDIO_EXTENSIONS = {".mp3", ".m4a", ".wav", ".flac", ".ogg", ".opus", ".aac", ".wma", ".mp4", ".webm", ".mov"}
ROOT = Path("/content")
RESULTS_ROOT = ROOT / "transcription_results"
RESULTS_ROOT.mkdir(exist_ok=True)

if INPUT_MODE == "upload":
    print("音声ファイルを選んでください（複数選択も可）")
    uploaded = files.upload()
    input_files = [ROOT / name for name in uploaded.keys()]
    del uploaded  # Google Colabのアップロード辞書が大きな音声のバイト列を保持するため解放
    gc.collect()
elif INPUT_MODE == "colab_existing":
    choices = sorted([p for p in ROOT.iterdir() if p.is_file() and p.suffix.lower() in AUDIO_EXTENSIONS],
                     key=lambda p: p.name.lower())
    if not choices:
        raise FileNotFoundError("/content 直下に音声ファイルがありません。ドラッグ＆ドロップしてください。")
    print("Colab内の音声ファイル一覧")
    for i, p in enumerate(choices, 1):
        print(f" {i}. {p.name}")
    answer = input("処理する番号を入力（例: 1、複数なら1,3、すべてならall）: ").strip()
    if answer.lower() == "all":
        input_files = choices
    else:
        try:
            indices = [int(x.strip()) for x in answer.split(",")]
            if not indices or any(i < 1 or i > len(choices) for i in indices):
                raise ValueError("番号が範囲外です。")
            input_files = [choices[i - 1] for i in dict.fromkeys(indices)]
        except ValueError as err:
            raise ValueError("一覧に表示された番号をカンマで区切って指定してください。") from err
else:
    raise ValueError(f"INPUT_MODEの設定が不正です: {INPUT_MODE}")

input_files = [p for p in input_files if p.suffix.lower() in AUDIO_EXTENSIONS and p.is_file()]
if not input_files:
    raise FileNotFoundError("対応している音声／動画ファイルがありません。MP3・M4A・WAV等を選択してください。")

# 元のファイル名と内容のハッシュからフォルダ名を作成し、同名の別音声の誤キャッシュを防ぐ。
def sha256_file(path):
    hasher = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            hasher.update(block)
    return hasher.hexdigest()

def safe_name(name):
    cleaned = re.sub(r'[\\/:*?"<>|\x00-\x1f]', "_", name).strip(" .")
    return cleaned[:90] or "audio"

jobs = []
for path in input_files:
    digest = sha256_file(path)
    prefix = f"{safe_name(path.stem)}_{digest[:10]}"
    outdir = RESULTS_ROOT / prefix
    outdir.mkdir(parents=True, exist_ok=True)
    jobs.append({"path": path, "sha256": digest, "prefix": prefix, "outdir": outdir})

print(f"入力ファイル: {len(jobs)}件")
for j in jobs:
    print(f" - {j['path'].name}  →  {j['outdir']}")

## ④ 話者分離モデルの認証（オンの場合のみ）
[Community-1の利用条件](https://huggingface.co/pyannote/speaker-diarization-community-1)への同意が必要です。Hugging Faceの[Readトークン](https://huggingface.co/settings/tokens)を作成してください。

毎回入力したくない場合、Colabの左側**鍵アイコン（シークレット）**に `HF_TOKEN` を保存し、このノートブックからのアクセスを許可すると自動読込します。トークンをコードに埋め込みません。

In [ ]:
HF_TOKEN = None
if ENABLE_DIARIZATION:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
    if not HF_TOKEN:
        from getpass import getpass
        HF_TOKEN = getpass("Hugging Face Readトークン（非表示入力）: ").strip()
    if not HF_TOKEN:
        raise ValueError("話者分離にはHugging Face Readトークンが必要です。不要ならENABLE_DIARIZATION=Falseにしてください。")
    from huggingface_hub import hf_hub_download
    try:
        hf_hub_download(repo_id="pyannote/speaker-diarization-community-1",
                        filename="config.yaml", token=HF_TOKEN)
    except Exception as err:
        raise RuntimeError("Community-1の利用条件への同意とHugging Face Readトークンを確認してください。") from err
    print("話者分離モデルの利用権限: OK")
else:
    print("話者分離はオフです（Hugging Face認証不要）。")

## ⑤ Whisper音声認識
選択した音声を順番に処理します。**元音声と認識設定が一致する場合のみ**、保存済みASR JSONを再利用します。途中経過は100区間ごとに一時保存しますが、途中停止から自動再開する機能ではありません。

In [ ]:
import json, gc
import torch, av
from faster_whisper import WhisperModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE = "float16" if DEVICE == "cuda" else "int8"
ASR_SETTINGS = {
    "model": WHISPER_MODEL,
    "language": LANGUAGE,
    "glossary": GLOSSARY.strip(),
    "beam_size": 5,
    "vad_filter": True,
    "min_silence_duration_ms": 550,
    "word_timestamps": True,
    "condition_on_previous_text": False,
}

pending = []
for job in jobs:
    dest = job["outdir"] / "whisper_asr.json"
    cached = None
    if dest.is_file():
        try:
            cached = json.loads(dest.read_text(encoding="utf-8"))
        except (ValueError, OSError):
            pass
    if (isinstance(cached, dict) and cached.get("source_sha256") == job["sha256"]
            and cached.get("settings") == ASR_SETTINGS and isinstance(cached.get("segments"), list)):
        job["asr_segments"] = cached["segments"]
        print(f"[ASR再利用] {job['path'].name}: {len(job['asr_segments'])}区間")
    else:
        pending.append(job)

if pending:
    print(f"Whisperモデルを読み込みます: {WHISPER_MODEL}, {DEVICE}/{COMPUTE}")
    model = WhisperModel(WHISPER_MODEL, device=DEVICE, compute_type=COMPUTE)
    for number, job in enumerate(pending, 1):
        print(f"\n[音声認識 {number}/{len(pending)}] {job['path'].name}")
        segments, info = model.transcribe(
            str(job["path"]),
            language=None if LANGUAGE == "auto" else LANGUAGE,
            task="transcribe",
            beam_size=5,
            vad_filter=True,
            vad_parameters={"min_silence_duration_ms": 550},
            initial_prompt=GLOSSARY.strip() or None,
            condition_on_previous_text=False,
            word_timestamps=True,
        )
        print(f"  言語:{info.language} / 録音:{info.duration / 60:.1f}分")
        rows = []
        partial = job["outdir"] / "whisper_asr_partial.json"
        for seg in segments:
            rows.append({
                "start": float(seg.start), "end": float(seg.end), "text": seg.text,
                "words": [
                    {"start": float(w.start), "end": float(w.end), "text": w.word}
                    for w in (seg.words or [])
                ],
            })
            if len(rows) % 100 == 0:
                partial.write_text(json.dumps(rows, ensure_ascii=False), encoding="utf-8")
                print(f"  認識済み: {seg.end / 60:.1f}分 / {len(rows)}区間")
        job["asr_segments"] = rows
        record = {"source_name": job["path"].name, "source_sha256": job["sha256"],
                  "settings": ASR_SETTINGS, "segments": rows}
        (job["outdir"] / "whisper_asr.json").write_text(
            json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
        partial.unlink(missing_ok=True)
        print(f"  認識完了: {len(rows)}区間")
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
print("全ファイルの音声認識が完了しました。")

## ⑥ 話者分離
話者分離がオンのときのみ実行します。ffmpegで16kHzのモノラルWAVに変換したうえでpyannoteに渡し、**非重複話者区間**を取得します。

同じ音声・同じ話者数設定の結果が残っていれば再利用します。変換した大きなWAVは処理後に削除します。

In [ ]:
import csv, gc, json, subprocess
import numpy as np
import soundfile as sf
import torch

DIAR_SETTINGS = {
    "model": "pyannote/speaker-diarization-community-1",
    "expected": int(EXPECTED_SPEAKERS),
    "min": int(MIN_SPEAKERS) if not EXPECTED_SPEAKERS else 0,
    "max": int(MAX_SPEAKERS) if not EXPECTED_SPEAKERS else 0,
}

if ENABLE_DIARIZATION:
    todo = []
    for job in jobs:
        dest = job["outdir"] / "pyannote_話者区間.json"
        cached = None
        if dest.exists():
            try:
                cached = json.loads(dest.read_text(encoding="utf-8"))
            except (ValueError, OSError):
                pass
        if (isinstance(cached, dict) and cached.get("source_sha256") == job["sha256"]
                and cached.get("settings") == DIAR_SETTINGS and isinstance(cached.get("segments"), list)):
            job["diar_segments"] = cached["segments"]
            print(f"[話者分離再利用] {job['path'].name}")
        else:
            todo.append(job)

    if todo:
        from pyannote.audio import Pipeline
        from pyannote.audio.pipelines.utils.hook import ProgressHook
        pipeline = Pipeline.from_pretrained(DIAR_SETTINGS["model"], token=HF_TOKEN)
        if torch.cuda.is_available():
            pipeline.to(torch.device("cuda"))
        diar_params = {}
        if EXPECTED_SPEAKERS:
            diar_params["num_speakers"] = int(EXPECTED_SPEAKERS)
        else:
            if MIN_SPEAKERS:
                diar_params["min_speakers"] = int(MIN_SPEAKERS)
            if MAX_SPEAKERS:
                diar_params["max_speakers"] = int(MAX_SPEAKERS)
        for number, job in enumerate(todo, 1):
            print(f"\n[話者分離 {number}/{len(todo)}] {job['path'].name}")
            wav_path = job["outdir"] / "temporary_16k_mono.wav"
            try:
                subprocess.run([
                    "ffmpeg", "-hide_banner", "-loglevel", "error", "-nostdin", "-y",
                    "-i", str(job["path"]), "-vn", "-ac", "1", "-ar", "16000",
                    "-c:a", "pcm_s16le", str(wav_path),
                ], check=True)
                samples, sample_rate = sf.read(wav_path, dtype="float32", always_2d=False)
                if sample_rate != 16000 or samples.ndim != 1:
                    raise ValueError("16kHz・モノラルへの変換に失敗しました。")
                waveform = torch.from_numpy(np.ascontiguousarray(samples)).unsqueeze(0)
                del samples
                with ProgressHook() as hook:
                    diar_output = pipeline({"waveform": waveform, "sample_rate": sample_rate},
                                           hook=hook, **diar_params)
                annotation = diar_output.exclusive_speaker_diarization
                intervals = [
                    {"start": float(turn.start), "end": float(turn.end), "speaker_id": str(speaker)}
                    for turn, _, speaker in annotation.itertracks(yield_label=True)
                ]
                intervals.sort(key=lambda x: (x["start"], x["end"]))
                job["diar_segments"] = intervals
                record = {"source_sha256": job["sha256"], "settings": DIAR_SETTINGS,
                          "segments": intervals}
                (job["outdir"] / "pyannote_話者区間.json").write_text(
                    json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
                print(f"  区間数: {len(intervals)} / 推定話者数: {len(set(x['speaker_id'] for x in intervals))}")
                del waveform, diar_output, annotation
                gc.collect()
                if DEVICE == "cuda":
                    torch.cuda.empty_cache()
            finally:
                wav_path.unlink(missing_ok=True)
        del pipeline
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    for job in jobs:
        path = job["outdir"] / "pyannote_話者区間.csv"
        with path.open("w", encoding="utf-8-sig", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=["start", "end", "speaker_id"])
            writer.writeheader()
            writer.writerows(job["diar_segments"])
else:
    for job in jobs:
        job["diar_segments"] = []
    print("話者分離は無効です。Whisperによる文字起こしのみ出力します。")

## ⑦ 単語時刻と話者を照合・発言単位にまとめる
話者分離ありの場合は、各単語の時間と重なりが最大の話者を採用します。区間外なら前後1.2秒以内の話者を探し、判定できなければ「話者不明」とします。

話者は登場順の番号（話者1、話者2…）であり、実際の役職や氏名ではありません。

In [ ]:
from bisect import bisect_right

def assign_and_group(asr_segments, diar_segments):
    ordered = sorted(diar_segments, key=lambda r: (r["start"], r["end"]))
    ends = [r["end"] for r in ordered]
    raw_to_label = {}
    for r in ordered:
        if r["speaker_id"] not in raw_to_label:
            raw_to_label[r["speaker_id"]] = f"話者{len(raw_to_label) + 1}"

    def speaker_for(start, end):
        if not ordered:
            return "話者未設定" if not ENABLE_DIARIZATION else "話者不明"
        start, end = float(start), max(float(start), float(end))
        # exclusive diarizationの区間は互いに重複せず、開始/終了時刻順でソート可能。
        idx = bisect_right(ends, start)
        scores = {}
        j = idx
        while j < len(ordered) and ordered[j]["start"] < max(start + 0.01, end):
            r = ordered[j]
            overlap = max(0.0, min(end, r["end"]) - max(start, r["start"]))
            if overlap > 0:
                scores[r["speaker_id"]] = scores.get(r["speaker_id"], 0.0) + overlap
            j += 1
        if scores:
            return raw_to_label[max(scores, key=scores.get)]
        center = (start + end) / 2.0
        near = ordered[max(idx - 1, 0):min(idx + 2, len(ordered))]
        if not near:
            return "話者不明"
        chosen = min(near, key=lambda r: max(r["start"] - center, 0, center - r["end"]))
        distance = max(chosen["start"] - center, 0, center - chosen["end"])
        return raw_to_label[chosen["speaker_id"]] if distance <= 1.2 else "話者不明"

    words = []
    for seg in asr_segments:
        for w in (seg.get("words") or []):
            if str(w.get("text", "")).strip():
                words.append({"start": float(w["start"]), "end": float(w["end"]),
                              "text": str(w["text"]), "speaker": speaker_for(w["start"], w["end"])})
        if not seg.get("words") and str(seg.get("text", "")).strip():
            words.append({"start": float(seg["start"]), "end": float(seg["end"]),
                          "text": str(seg["text"]), "speaker": speaker_for(seg["start"], seg["end"])})
    words.sort(key=lambda r: (r["start"], r["end"]))
    MAX_CHARS, MAX_SECONDS, MAX_PAUSE = 110, 22, 1.5
    utterances, current = [], None
    for w in words:
        if current is None:
            current = {"start": w["start"], "end": w["end"], "speaker": w["speaker"], "parts": [w["text"]]}
            continue
        pause = w["start"] - current["end"]
        too_long = ((w["end"] - current["start"] > MAX_SECONDS)
                    or (sum(map(len, current["parts"])) + len(w["text"]) > MAX_CHARS))
        if w["speaker"] != current["speaker"] or pause > MAX_PAUSE or too_long:
            content = "".join(current["parts"]).strip()
            if content:
                utterances.append({"start": current["start"], "end": current["end"],
                                   "speaker": current["speaker"], "text": content})
            current = {"start": w["start"], "end": w["end"], "speaker": w["speaker"], "parts": [w["text"]]}
        else:
            current["end"] = max(current["end"], w["end"])
            current["parts"].append(w["text"])
    if current:
        content = "".join(current["parts"]).strip()
        if content:
            utterances.append({"start": current["start"], "end": current["end"],
                               "speaker": current["speaker"], "text": content})
    return utterances, raw_to_label, len(words)

for job in jobs:
    job["utterances"], job["speaker_map"], job["word_count"] = assign_and_group(
        job["asr_segments"], job["diar_segments"])
    print(f"{job['path'].name}: {len(job['utterances'])}発言、{job['word_count']}単語相当区間、"
          f"{len(job['speaker_map'])}推定話者")

## ⑧ TXT・CSV・SRT・JSONをZIPにまとめ、ダウンロード
元の音声はZIPに含めません。2ファイル以上のときは、結果ZIPをひとつに束ねてダウンロードします。`SAVE_TO_DRIVE=True` にした場合のみ、ZIPをGoogle Driveにもコピーします。

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
from pathlib import Path
import csv, json, shutil

def hhmmss(sec):
    sec = max(0, int(round(sec)))
    return f"{sec // 3600:02d}:{sec % 3600 // 60:02d}:{sec % 60:02d}"

def srt_ts(sec):
    ms = max(0, int(round(sec * 1000)))
    return f"{ms // 3600000:02d}:{ms % 3600000 // 60000:02d}:{ms % 60000 // 1000:02d},{ms % 1000:03d}"

zip_files = []
for job in jobs:
    outdir, prefix = job["outdir"], job["prefix"]
    rows, speakers = job["utterances"], job["speaker_map"]
    txt = outdir / f"{prefix}_文字起こし.txt"
    csv_path = outdir / f"{prefix}_文字起こし.csv"
    srt = outdir / f"{prefix}_文字起こし.srt"
    mapping = outdir / "話者IDの対応表.csv"
    with txt.open("w", encoding="utf-8") as f:
        f.write(f"音声: {job['path'].name} / 機械文字起こし（要確認）\n")
        f.write("話者番号・時刻は自動推定です。役職・氏名は未特定。\n\n")
        for row in rows:
            f.write(f"[{hhmmss(row['start'])} - {hhmmss(row['end'])}] {row['speaker']}：{row['text']}\n")
    with csv_path.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["start", "end", "speaker", "utterance"])
        for row in rows:
            writer.writerow([hhmmss(row["start"]), hhmmss(row["end"]), row["speaker"], row["text"]])
    with srt.open("w", encoding="utf-8") as f:
        for i, row in enumerate(rows, 1):
            f.write(f"{i}\n{srt_ts(row['start'])} --> {srt_ts(row['end'])}\n{row['speaker']}：{row['text']}\n\n")
    with mapping.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["モデル話者ID", "文字起こしの表示", "役職や氏名（手動記入）"])
        for raw_id, label in speakers.items():
            writer.writerow([raw_id, label, ""])
    report = {
        "source_filename": job["path"].name,
        "source_sha256": job["sha256"],
        "asr_settings": ASR_SETTINGS,
        "diarization_enabled": ENABLE_DIARIZATION,
        "diarization_settings": DIAR_SETTINGS if ENABLE_DIARIZATION else None,
        "asr_segments": len(job["asr_segments"]),
        "word_intervals": job["word_count"],
        "diarization_intervals": len(job["diar_segments"]),
        "estimated_speakers": len(speakers),
        "utterances": len(rows),
        "note": "機械処理結果。発言者の同一性、事業名、発言内容は必ず原音声と照合してください。",
    }
    report_path = outdir / "処理レポート.json"
    report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    outputs = [txt, csv_path, srt, mapping, outdir / "whisper_asr.json", report_path]
    if ENABLE_DIARIZATION:
        outputs.extend([outdir / "pyannote_話者区間.csv", outdir / "pyannote_話者区間.json"])
    zip_path = ROOT / f"{prefix}_文字起こし結果.zip"
    with ZipFile(zip_path, "w", ZIP_DEFLATED, compresslevel=6) as z:
        for p in outputs:
            z.write(p, arcname=p.name)
    job["zip_path"] = zip_path
    zip_files.append(zip_path)
    print(f"作成: {zip_path.name} / {round(zip_path.stat().st_size / 1024**2, 2)} MiB")

if len(zip_files) == 1:
    download_target = zip_files[0]
else:
    download_target = ROOT / "文字起こし結果_一括.zip"
    with ZipFile(download_target, "w", ZIP_DEFLATED, compresslevel=0) as z:
        for p in zip_files:
            z.write(p, arcname=p.name)

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_dir = Path("/content/drive/MyDrive") / DRIVE_OUTPUT_FOLDER
    drive_dir.mkdir(parents=True, exist_ok=True)
    for p in zip_files:
        shutil.copy2(p, drive_dir / p.name)
    print("Google Drive保存先:", drive_dir)

print("\nダウンロード:", download_target.name)
files.download(str(download_target))

---
### 再実行・注意事項
- **別の音声を処理するとき**：通常は③以降を再実行し、新しい音声を選びます。モデルや話者数などを変えたら②の設定を変えて③以降を実行してください。
- **セッション再起動後**：Colabの `/content` は一時領域なので、音声や中間JSONが消える場合があります。必要な結果は最後のZIP（またはDrive）に保存してください。
- **同じ音声を再処理するとき**：`/content/transcription_results/` に結果が残っていれば、音声の内容ハッシュと設定を照合してASR・話者分離を再利用します。
- **話者数が多すぎるとき**：②の `EXPECTED_SPEAKERS` に実際の参加者数を指定して試してください。ただし固定人数を指定しても識別が正確になるとは限りません。
- **長時間録音の品質**：発言が重なる部分、短い相づち、遠くの声、固有名詞は原音声との照合が必要です。
- **セキュリティ**：Hugging Faceのトークンや会議音声を共有ノートブックに埋め込まないでください。所属組織の情報セキュリティ・クラウド利用規程を優先してください。